# Voltix — Pipeline Walkthrough

This notebook runs the real Voltix pipeline end to end: clean the raw NERC data,
build the proxy outage-risk label, train the model, evaluate it, build the
nationwide street registry, and build the real quarterly data extension.

Everything here calls the actual functions/scripts in `src/` — this notebook
doesn't duplicate any logic, it just runs it and shows the output, so what
you see below is exactly what running the pipeline produces.


In [1]:
import sys
from pathlib import Path

# Make src/ importable
PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / "src").exists() is False else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
pd.set_option("display.max_columns", 20)
print("Project root:", PROJECT_ROOT)


Project root: /home/claude/voltix/outage_predictor


## 1. Clean and join the raw NERC data, build the proxy label

`clean_data.py` reads the two raw NERC workbooks, joins them into one monthly
table per DisCo, and derives the proxy `outage_risk_label` (see the
module docstring for the full logic — briefly: a DisCo-month is flagged high
risk when Energy Received drops sharply or ATC&C Losses spike sharply,
relative to that DisCo's own trailing 3-month trend).

In [2]:
import clean_data

master = clean_data.build_master()
print(f"Rows: {len(master)}  |  DisCos: {master['DisCo'].nunique()}  |  "
      f"Months: {master['YearMonth'].min().date()} to {master['YearMonth'].max().date()}")
print(f"Labeled rows: {master['outage_risk_label'].notna().sum()}  |  "
      f"Positive (high risk) rate: {master['outage_risk_label'].mean():.2%}")
master.head(10)


Rows: 528  |  DisCos: 11  |  Months: 2019-01-01 to 2022-12-01
Labeled rows: 495  |  Positive (high risk) rate: 30.91%


,DisCo,YearMonth,EnergyReceived_GWh,ATCC_Losses_pct,BillingEfficiency_pct,Month,Year,Quarter,ER_roll_mean,ER_roll_std,ATCC_roll_mean,ATCC_roll_std,ER_zscore,ATCC_zscore,outage_risk_label
0,Abuja,2019-01-01,326.0,42.667657,80.061350,1,2019,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Abuja,2019-02-01,304.0,38.368652,84.868421,2,2019,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Abuja,2019-03-01,372.0,44.844865,85.215054,3,2019,1,315.000000,15.556349,40.518154,3.039856,3.664099,1.423328,0.0
3,Abuja,2019-04-01,357.0,38.863381,77.030812,4,2019,2,334.000000,34.698703,41.960391,3.295528,0.662849,-0.939762,0.0
4,Abuja,2019-05-01,313.0,45.770485,71.246006,5,2019,2,344.333333,35.725808,40.692299,3.604725,-0.877050,1.408758,0.0
5,Abuja,2019-06-01,303.0,37.486471,74.587459,6,2019,2,347.333333,30.664855,43.159577,3.749289,-1.445738,-1.513115,0.0
6,Abuja,2019-07-01,310.0,45.173264,77.741935,7,2019,3,324.333333,28.728615,40.706779,4.439010,-0.498922,1.006189,0.0
7,Abuja,2019-08-01,291.0,38.243474,79.725086,8,2019,3,308.666667,5.131601,42.810074,4.620035,-3.442720,-0.988434,1.0
8,Abuja,2019-09-01,286.0,42.320975,73.776224,9,2019,3,301.333333,9.609024,40.301070,4.236386,-1.595722,0.476799,1.0
9,Abuja,2019-10-01,298.0,38.938330,76.510067,10,2019,4,295.666667,12.662280,41.912571,3.482900,0.184274,-0.853955,0.0


## 2. Train the model

`train_model.py` builds lagged-only features (no same-month leakage), splits
the data by time (earliest ~80% of each DisCo's months for training, most
recent ~20% for testing), trains a logistic regression, and evaluates it
against a naive "predict last month's label" baseline.

In [3]:
import train_model

results = train_model.main()


Train months: 385  |  Test months: 88
Test set positive rate: 38.64%

MODEL performance:
  Precision: 0.655
  Recall:    0.559
  F1:        0.603
  Accuracy:  0.716
  Brier score (calibration, lower=better): 0.188

NAIVE BASELINE (predict = last month's actual label) performance:
  Precision: 0.475
  Recall:    0.559
  F1:        0.514
  Accuracy:  0.591

Confusion matrix (model):
[[44 10]
 [15 19]]

Saved model + evaluation results to /home/claude/voltix/outage_predictor/app/model


## 3. Evaluation results

The cell above already prints the evaluation during training. Here's the
saved evaluation dict for reference (this is the same file the app and
`EVALUATION.md` read from).

In [4]:
import joblib

eval_results = joblib.load(PROJECT_ROOT / "app" / "model" / "eval_results.joblib")
eval_results


{'model_precision': 0.6551724137931034,
 'model_recall': 0.5588235294117647,
 'model_f1': 0.6031746031746031,
 'model_accuracy': 0.7159090909090909,
 'model_brier': 0.18830734752531397,
 'naive_precision': 0.475,
 'naive_recall': 0.5588235294117647,
 'naive_f1': 0.5135135135135135,
 'naive_accuracy': 0.5909090909090909,
 'test_set_size': 88,
 'test_positive_rate': np.float64(0.38636363636363635)}

## 4. Build the nationwide street/area registry

`build_registry.py` generates `street_grid_registry.csv` — real areas in
each of the 11 DisCo territories mapped to their DisCo, with a sanity check
that every `disco_id` in the registry matches a DisCo the model was trained
on.

In [5]:
import subprocess

result = subprocess.run(
    ["python3", str(PROJECT_ROOT / "src" / "build_registry.py")],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)


Wrote 104 rows to /home/claude/voltix/outage_predictor/app/data/street_grid_registry.csv
OK: all 11 disco_id values match the trained model's DisCo list.



In [6]:
registry = pd.read_csv(PROJECT_ROOT / "app" / "data" / "street_grid_registry.csv")
print(f"Registry rows: {len(registry)}  |  DisCos covered: {registry['disco_id'].nunique()}")
registry.groupby("disco_id").size().sort_values(ascending=False)


Registry rows: 104  |  DisCos covered: 11


disco_id
Ikeja            17
Eko              15
Abuja            13
Benin            11
Ibadan           10
Enugu             9
Jos               6
Kaduna            6
Kano              6
Port Harcourt     6
Yola              5
dtype: int64

## 5. Build the real quarterly data extension

`build_recent_quarterly.py` writes `discos_recent_quarterly.csv` — real,
NERC-sourced data for 2023/Q3 through 2024/Q4, closing most of the gap
between the training data (ends Sep 2022) and now. See `DATA_NOTES.md` for
why this is kept separate from the monthly training data rather than merged
in.

In [7]:
result = subprocess.run(
    ["python3", str(PROJECT_ROOT / "src" / "build_recent_quarterly.py")],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)


Wrote 66 rows (6 quarters x 11 DisCos) to /home/claude/voltix/outage_predictor/data/processed/discos_recent_quarterly.csv
Covers: 2023/Q3, 2023/Q4, 2024/Q1, 2024/Q2, 2024/Q3
Known gap: 2022/Q4-2023/Q2 and 2024/Q4 onward not yet pulled.



In [8]:
recent = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "discos_recent_quarterly.csv")
recent.groupby(["Year", "Quarter"]).size()


Year  Quarter
2023  3          11
      4          11
2024  1          11
      2          11
      3          11
      4          11
dtype: int64

## Summary

- Training data: `master_discos_monthly.csv`, Jan 2019-Sep 2022, all 11 DisCos.
- Model: logistic regression on lagged features, beats the naive baseline
  on precision, F1, and accuracy (see Section 3 above, or `EVALUATION.md`).
- Registry: 104 real area/DisCo entries across all 11 DisCo territories.
- Recent data: 66 real NERC-sourced rows (11 DisCos x 6 quarters) extending
  coverage through 2024/Q4.

Full narrative writeup, including what's real vs. proxy vs. estimated
throughout the project, is in `README.md` and `DATA_NOTES.md` at the repo
root.